In [1]:
import os.path

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

import scienceplots
plt.style.use(['science', "no-latex"])

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
ITL3_region = gpd.read_file(r"./results/intermediate/ITL3_region.gpkg")
substations = gpd.read_file(r"./results/intermediate/substations.gpkg")

In [3]:
interested_locations = {
    # "London": {},
    "TLI3": {},
    "TLI4": {},
    "TLI5": {},
    "TLI6": {},
    "TLI7": {},
    "TLH1": {},
    "TLH2": {},
    "TLH3": {},
    "TLJ1":{},
    "TLF1":{},
    "TLF2":{},
    "TLC1":{},
    "TLC2":{},
    "TLD3":{},
    "TLD4":{},
    "TLD6":{},
    "TLG1":{},
    "TLG2":{},
    "TLE3":{},
    "TLE4":{},
}

In [4]:
for itl2_code in interested_locations.keys():
    interested_region = ITL3_region[ITL3_region['ITL2'] == itl2_code]
    interested_substations = substations[substations['ITL2'] == itl2_code]

    interested_locations[itl2_code] = {
        "region": interested_region.reset_index(),
        "substation": interested_substations.reset_index()
    }

# Grid Points

In [5]:
from SpatialAllocation.utils.GenerateGrid import generate_grid
from shapely.ops import unary_union
import pickle

In [6]:
def calculate_step_size(target_points, regions):
    """
    计算合适的网格步长
    :param target_points: 目标点数
    :param regions: 地区的GeoDataFrame
    :return: 合适的网格步长
    """
    regions = regions.to_crs("EPSG:3857")
    boundary_polygon_m = unary_union(regions["geometry"])
    minx, miny, maxx, maxy = boundary_polygon_m.bounds
    box_area = (maxx - minx) * (maxy - miny)
    polygon_area = boundary_polygon_m.area
    area_ratio = polygon_area / box_area
    adjusted_target = target_points / area_ratio
    ideal_step_size = np.sqrt(box_area / adjusted_target)
    step_size_m = np.floor(ideal_step_size / 10) * 10
    return max(10, step_size_m)

In [7]:
numerical_col_names_all = []
categorical_col_members_all = {}

In [8]:
for key in interested_locations.keys():
    grid_gdf_path = f"./results/intermediate/GridPoint_GNN/{key}_grid_points.pickle"
    print("Processing interested location:", key)
    if os.path.exists(grid_gdf_path):
        with open(grid_gdf_path, "rb") as f:
            grid_gdf, numerical_col_names, categorical_col_members, step_size_m = pickle.load(f)
            print(f"step_size_m for {key} is {step_size_m} m")
    else:
        step_size_m = calculate_step_size(target_points=200000, regions=interested_locations[key]["region"].copy())
        # step_size_m = 1000
        print(f"Calculated step_size_m for {key} is {step_size_m} m")
        grid_gdf, numerical_col_names, categorical_col_members = generate_grid(interested_locations[key]["region"].copy(), step_size_m=step_size_m, with_tags={"landuse":True})
        print(grid_gdf.shape)

        with open(grid_gdf_path, "wb") as f:
            pickle.dump([grid_gdf, numerical_col_names, categorical_col_members, step_size_m], f)

    interested_locations[key]["grid"] = grid_gdf
    interested_locations[key]["step_size_m"] = step_size_m

    numerical_col_names_all = list(set(numerical_col_names_all + numerical_col_names))
    for cat_key, cat_members in categorical_col_members.items():
        existing_members = categorical_col_members_all.get(cat_key, [])
        updated_members = list(set(existing_members + cat_members))
        categorical_col_members_all[cat_key] = updated_members

print("\n--- Aggregation Results ---")
print("All unique numerical column names:", numerical_col_names_all)
print("All unique categorical column members:", categorical_col_members_all)

Processing interested location: TLI3
step_size_m for TLI3 is 70.0 m
Processing interested location: TLI4
step_size_m for TLI4 is 100.0 m
Processing interested location: TLI5
step_size_m for TLI5 is 140.0 m
Processing interested location: TLI6
step_size_m for TLI6 is 130.0 m
Processing interested location: TLI7
step_size_m for TLI7 is 150.0 m
Processing interested location: TLH1
step_size_m for TLH1 is 820.0 m
Processing interested location: TLH2
step_size_m for TLH2 is 380.0 m
Processing interested location: TLH3
step_size_m for TLH3 is 430.0 m
Processing interested location: TLJ1
step_size_m for TLJ1 is 540.0 m
Processing interested location: TLF1
step_size_m for TLF1 is 510.0 m
Processing interested location: TLF2
step_size_m for TLF2 is 510.0 m
Processing interested location: TLC1
step_size_m for TLC1 is 420.0 m
Processing interested location: TLC2
step_size_m for TLC2 is 580.0 m
Processing interested location: TLD3
step_size_m for TLD3 is 260.0 m
Processing interested location: TLD

In [9]:
with open(f"./results/intermediate/GridPoint_GNN/all_node_features_col.pickle", "wb") as f:
    pickle.dump([numerical_col_names_all, categorical_col_members_all], f)